# 02 — Label Quality Analysis

Validates the label engineering decisions made in `src/data/label_engineering.py`:

1. Priority normalisation — confusion between original severity and normalised P0–P4
2. Team label coverage — what % ends up as `unknown`; spot-check 200 samples
3. Over-escalation label validation — positive/negative side-by-side examples
4. Hedging word frequency analysis — which words drive the Eclipse heuristic
5. Cross-source label consistency check

**Run `src/data/load_datasets.py` first.**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PROCESSED = '../data/processed'

In [ ]:
train = pd.read_parquet(f'{PROCESSED}/train.parquet')
val   = pd.read_parquet(f'{PROCESSED}/val.parquet')
test  = pd.read_parquet(f'{PROCESSED}/test.parquet')
df    = pd.concat([train, val, test], ignore_index=True)
print(f'{len(df):,} total records | columns: {list(df.columns)}')

## 1. Priority normalisation — Eclipse source

Check that the Eclipse `raw_severity → priority` mapping is correct.

In [ ]:
eclipse = df[df['source'] == 'eclipse'].copy()

if 'raw_severity' in eclipse.columns and eclipse['raw_severity'].any():
    cross = pd.crosstab(eclipse['raw_severity'], eclipse['priority'])
    print('Eclipse raw_severity → normalised priority cross-tab:')
    display(cross)

    # Verify the mapping is bijective (each severity maps to exactly one priority)
    for sev, row in cross.iterrows():
        mapped_to = row[row > 0].index.tolist()
        if len(mapped_to) != 1:
            print(f'⚠️  {sev!r} maps to multiple priorities: {mapped_to}')
        else:
            print(f'✓  {sev!r} → {mapped_to[0]}')
else:
    print('raw_severity column not available or empty.')

## 2. Team label coverage — spot check 200 samples

In [ ]:
# Overall team distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

team_dist = df['team'].value_counts()
team_dist.plot.bar(ax=axes[0], color=sns.color_palette('Set2', len(team_dist)), edgecolor='white')
axes[0].set_title('Team label distribution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Coverage by source
coverage = df.groupby('source').apply(
    lambda g: 100 * (g['team'] != 'unknown').mean()
)
coverage.plot.bar(ax=axes[1], color=sns.color_palette('muted'), edgecolor='white')
axes[1].set_title('Team label coverage by source (% non-unknown)')
axes[1].set_ylabel('% covered')
axes[1].tick_params(axis='x', rotation=0)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.savefig('../results/team_coverage.png')
plt.show()

# Spot check 200 random samples
spot = df.sample(200, random_state=42)[['source', 'title', 'body', 'team', 'priority']]
spot['body_preview'] = spot['body'].str[:100]
print('\n200-sample spot check (review team assignments manually):')
display(spot[['source', 'title', 'team', 'priority', 'body_preview']].head(20))

## 3. Over-escalation — positive vs negative examples side by side

In [ ]:
def show_pairs(source: str, n: int = 3):
    src_df = df[df['source'] == source]
    pos = src_df[src_df['is_over_escalated']].sample(min(n, len(src_df[src_df['is_over_escalated']])), random_state=42)
    neg = src_df[~src_df['is_over_escalated']].sample(min(n, len(src_df[~src_df['is_over_escalated']])), random_state=42)

    print(f'\n{"="*70}')
    print(f'SOURCE: {source.upper()}')
    print(f'{"="*70}')

    for label, subset in [('OVER-ESCALATED (positive)', pos), ('NOT over-escalated (negative)', neg)]:
        print(f'\n  ── {label} ──')
        for _, row in subset.iterrows():
            esc_flag = '🔴' if row['is_over_escalated'] else '🟢'
            print(f'  {esc_flag}  [{row["priority"]}] {row["title"]}')
            body = row['body'][:200] if row['body'] else '(no body)'
            if body:
                print(f'     {body[:120]}')
            if 'resolution' in row and row['resolution']:
                print(f'     Resolution: {row["resolution"]}  |  Time: {row.get("resolution_time_days", "N/A")} days')
            print()

for src in df['source'].unique():
    show_pairs(src)

## 4. Hedging word frequency analysis (Eclipse heuristic)

In [ ]:
HEDGING_WORDS = [
    'sometimes', 'occasionally', 'intermittent', 'might', 'maybe',
    'appears to', 'seems', 'randomly', 'flaky', 'not reproducible',
    'not always', 'sporadically', 'unclear', 'not sure', 'possibly',
    'perhaps',
]

eclipse = df[df['source'] == 'eclipse'].copy()
eclipse_high = eclipse[eclipse['priority'].isin(['P0', 'P1'])].copy()

if len(eclipse_high) > 0:
    text_col = (eclipse_high['title'] + ' ' + eclipse_high['body']).str.lower()
    freq = {w: text_col.str.contains(re.escape(w), regex=False).sum() for w in HEDGING_WORDS}
    freq_df = pd.Series(freq).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(8, 4))
    freq_df[freq_df > 0].plot.barh(ax=ax, color='#ed7d31')
    ax.set_title(f'Hedging word occurrences in Eclipse P0/P1 tickets ({len(eclipse_high):,} total)')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.savefig('../results/hedging_words.png')
    plt.show()

    print('Top hedging words:')
    print(freq_df.to_string())
else:
    print('No Eclipse P0/P1 records found.')

## 5. Cross-source consistency check

Ensure that label distributions are broadly similar across sources
so mixing them doesn't introduce systematic bias.

In [ ]:
print('Priority distribution by source (%):')
display(
    df.groupby('source')['priority']
      .value_counts(normalize=True)
      .mul(100)
      .round(1)
      .unstack(fill_value=0)
      .reindex(columns=['P0','P1','P2','P3','P4'], fill_value=0)
)

print('\nTeam distribution by source (%):')
display(
    df.groupby('source')['team']
      .value_counts(normalize=True)
      .mul(100)
      .round(1)
      .unstack(fill_value=0)
)

print('\nOver-escalation rate by source:')
display(
    df.groupby('source')['is_over_escalated']
      .agg(['mean', 'sum', 'count'])
      .assign(rate_pct=lambda x: (x['mean']*100).round(1))
      .drop(columns='mean')
      .rename(columns={'sum': 'positives', 'count': 'total', 'rate_pct': 'rate %'})
)

## 6. Final assessment

Fill this in after reviewing the outputs above.

In [ ]:
esc_rate = 100 * df['is_over_escalated'].mean()
unknown_rate = 100 * (df['team'] == 'unknown').mean()

checks = {
    'Over-escalation rate in [2%, 25%]'   : 2 <= esc_rate <= 25,
    'Unknown team rate < 40%'              : unknown_rate < 40,
    'All 5 priorities represented'         : df['priority'].nunique() == 5,
    'Esc. positives in train split'        : train['is_over_escalated'].any(),
    'Esc. positives in val split'          : val['is_over_escalated'].any(),
    'Esc. positives in test split'         : test['is_over_escalated'].any(),
}

print('Label quality checks:')
for check, passed in checks.items():
    icon = '✅' if passed else '❌'
    print(f'  {icon}  {check}')

print(f'\nOver-escalation rate : {esc_rate:.1f}%')
print(f'Unknown team rate    : {unknown_rate:.1f}%')